# Bronze Layer - Performance Optimizations

This notebook implements the following Spark performance optimizations:

## 1. **Repartition vs Coalesce**
- **Repartition(8)**: Used during processing to distribute data evenly across 8 partitions for better parallelism
- **Coalesce(4)**: Used before writes to reduce small files (initial load)
- **Coalesce(2)**: Used for incremental writes to minimize file fragmentation
- **Why?** Repartition shuffles data for balanced processing; coalesce reduces partitions without shuffle for efficient writes

## 2. **Partitioning Strategy**
- Tables partitioned by `created_at` (date column)
- **Benefits**: 
  - Faster time-based queries (e.g., last 30 days)
  - Efficient data pruning during reads
  - Better MERGE performance for incremental loads

## 3. **Small Files Problem**
- Coalesce before writes prevents creation of many small files
- OPTIMIZE command at the end compacts files
- **Impact**: Reduces metadata overhead and improves read performance

In [0]:
#import statements
import json
from pyspark.sql.functions import *


In [0]:
dbutils.widgets.text("catalog","dev1")

In [0]:
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "dev1"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA bronze")

# Create control table if not exists
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.bronze.ingestion_control (
        dataset_name STRING,
        last_run_time TIMESTAMP,
        PRIMARY KEY (dataset_name)
    )
""")

config = json.load(open("/Workspace/Users/polasaipranav@gmail.com/databricks_assessment/assessment1/config_files/config.json"))

for d in config["datasets"]:
    dataset_name = d['name']
    primary_key = d['primary_key']
    target_table = f"{catalog}.bronze.bronze_{dataset_name}"
    
    print(f"\nProcessing: {dataset_name}")
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(target_table)
    
    # Get last run time if table exists
    last_run_time = None
    if table_exists:
        result = spark.sql(f"""
            SELECT last_run_time 
            FROM {catalog}.bronze.ingestion_control 
            WHERE dataset_name = '{dataset_name}'
        """).collect()
        if result:
            last_run_time = result[0]['last_run_time']
    
    # Read source data
    df = spark.read.csv(f"/Volumes/{catalog}/bronze/raw/{dataset_name}.csv", header=True, inferSchema=True)
    
    # OPTIMIZATION 1: Repartition for better parallelism during processing
    # Use 8 partitions for balanced parallelism
    df = df.repartition(8)
    
    # Add metadata columns
    current_time = current_timestamp()
    df = df.withColumn("ingestion_time", current_time)
    df = df.withColumn("path", col("_metadata.file_path"))
    
    # Initial load or incremental load
    if not table_exists:
        print(f"  Initial load for {dataset_name}")
        # OPTIMIZATION 2: Coalesce before write to reduce small files
        # Coalesce to 4 files for initial load
        df = df.coalesce(4)
        
        # OPTIMIZATION 3: Partition by date for time-based queries
        df.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .partitionBy("created_at") \
            .saveAsTable(target_table)
    else:
        print(f"  Incremental load for {dataset_name}")
        # Incremental load: filter by created_at or updated_at
        if last_run_time:
            # Filter records that are new or updated since last run
            df_incremental = df.filter(
                (col("created_at") > lit(last_run_time)) | 
                (col("updated_at") > lit(last_run_time))
            )
        else:
            df_incremental = df
        
        # OPTIMIZATION 4: Coalesce incremental data to avoid small files
        df_incremental = df_incremental.coalesce(2)
        
        # Create temp view for MERGE
        df_incremental.createOrReplaceTempView(f"temp_{dataset_name}")
        
        # MERGE (UPSERT) logic
        spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING temp_{dataset_name} AS source
            ON target.{primary_key} = source.{primary_key}
            WHEN MATCHED THEN
                UPDATE SET *
            WHEN NOT MATCHED THEN
                INSERT *
        """)
    
    # Update control table with current run time
    spark.sql(f"""
        MERGE INTO {catalog}.bronze.ingestion_control AS target
        USING (SELECT '{dataset_name}' as dataset_name, current_timestamp() as last_run_time) AS source
        ON target.dataset_name = source.dataset_name
        WHEN MATCHED THEN
            UPDATE SET last_run_time = source.last_run_time
        WHEN NOT MATCHED THEN
            INSERT (dataset_name, last_run_time) VALUES (source.dataset_name, source.last_run_time)
    """)
    
    print(f"  ✓ Completed ingestion for {dataset_name}")

print("\n" + "="*60)
print("Bronze layer ingestion complete!")
print("="*60)

In [0]:
# 🔹 Optimize Bronze
for d in config["datasets"]:
   spark.sql(f"OPTIMIZE {catalog}.bronze.bronze_{d['name']}")